# Churn Prediction

This notebook builds the predictive baseline for **Beyond Churn**.

The question:

> **Who is most likely to churn without a retention treatment?**

Because the dataset comes from an experiment, I train the churn model only on the **control group**. That ensures the model is focused on just untreated churn risk.

The workflow is kept simple. I build a logistic regression baseline, compare it with CatBoost, choose a threshold using out-of-fold predictions, and evaluate on a held-out test set.

In [1]:
# imports
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openml

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_val_predict
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay
)

from catboost import CatBoostClassifier

RANDOM_STATE = 42

## 1. Load and prepare the data

Notebook 01 contains the full data audit

In [2]:
dataset = openml.datasets.get_dataset(45580)

X_raw, y, _, _ = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute
)

df = X_raw.copy()
df["y"] = y

treatment = df["t"].astype(int)
y = df["y"].astype(int)

X = df.drop(columns=["t", "y"]).copy()

# Constant feature identified in Notebook 01
if "FACTOR3" in X.columns:
    X = X.drop(columns=["FACTOR3"])

categorical_cols = X.select_dtypes(
    include=["object", "string", "str", "category"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

X[categorical_cols] = X[categorical_cols].astype(str)

print("Rows:", len(X))
print("Features:", X.shape[1])
print("Overall churn rate:", round(y.mean(), 4))

Rows: 11896
Features: 177
Overall churn rate: 0.0343


## 2. Use the control group

The treatment itself can change churn, so I exclude treated customers from the predictive model.

This gives the model a cleaner interpretation, specifically **risk of churn without intervention**.

In [3]:
control_mask = treatment == 0

X_risk = X.loc[control_mask].copy()
y_risk = y.loc[control_mask].copy()

print("Control customers:", len(X_risk))
print("Control churners:", int(y_risk.sum()))
print("Control churn rate:", round(y_risk.mean(), 4))

Control customers: 2886
Control churners: 105
Control churn rate: 0.0364


## 3. Train / test split

The test set is set aside until the end. Model selection and threshold selection use only the training data.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_risk,
    y_risk,
    test_size=0.25,
    stratify=y_risk,
    random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "Rows": [len(y_train), len(y_test)],
    "Churners": [int(y_train.sum()), int(y_test.sum())],
    "Churn rate": [y_train.mean(), y_test.mean()]
}, index=["Train", "Test"])

split_summary

,Rows,Churners,Churn rate
Train,2164,79,0.036506
Test,722,26,0.036011


## 4. Logistic regression baseline

Logistic regression is a good baseline before trying a more flexible or advanced tree-based model.

Numeric features are standardized and categorical features are one-hot encoded inside the pipeline.

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_cols),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

logistic_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

logistic_oof = cross_val_predict(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF ROC AUC:", round(roc_auc_score(y_train, logistic_oof), 3))
print(
    "OOF Average Precision:",
    round(average_precision_score(y_train, logistic_oof), 3)
)

OOF ROC AUC: 0.792
OOF Average Precision: 0.145


The out-of-fold scores give each training row a prediction from a model that did not train on that row. And this gives us a fairer estimate for model comparison.

## 5. CatBoost challenger

CatBoost is great at handling categorical features. So, it is a good fit for this dataset.

I use a small hyperparameter search. The goal is to find a reasonable model, not try to exhaust every possible configuration.

In [6]:
catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    verbose=2,
    random_seed=RANDOM_STATE,
    thread_count=-1
)

catboost_params = {
    "iterations": [150, 300],
    "depth": [4, 6],
    "learning_rate": [0.03, 0.10],
    "l2_leaf_reg": [3, 10]
}

catboost_search = RandomizedSearchCV(
    estimator=catboost_model,
    param_distributions=catboost_params,
    n_iter=5,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

catboost_search.fit(
    X_train,
    y_train,
    cat_features=categorical_cols
)

print("Best parameters:", catboost_search.best_params_)
print(
    "Best CV Average Precision:",
    round(catboost_search.best_score_, 3)
)

KeyboardInterrupt: 

## 6. Compare the models with out-of-fold predictions

I generate CatBoost out-of-fold predictions using the selected parameters, then compare both models using the same training folds.

In [ ]:
best_catboost_params = catboost_search.best_params_

catboost_oof = np.zeros(len(X_train))

for train_fold_idx, valid_fold_idx in cv.split(X_train, y_train):
    fold_model = CatBoostClassifier(
        **best_catboost_params,
        loss_function="Logloss",
        eval_metric="PRAUC",
        verbose=0,
        random_seed=RANDOM_STATE,
        thread_count=-1
    )

    fold_model.fit(
        X_train.iloc[train_fold_idx],
        y_train.iloc[train_fold_idx],
        cat_features=categorical_cols
    )

    catboost_oof[valid_fold_idx] = fold_model.predict_proba(
        X_train.iloc[valid_fold_idx]
    )[:, 1]

model_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "CatBoost"],
    "ROC AUC": [
        roc_auc_score(y_train, logistic_oof),
        roc_auc_score(y_train, catboost_oof)
    ],
    "Average Precision": [
        average_precision_score(y_train, logistic_oof),
        average_precision_score(y_train, catboost_oof)
    ]
})

model_comparison

CatBoost is selected as the final churn-risk model based mainly on **Average Precision**, which is useful here because churn is rare.

The CatBoost output is used as a **risk score for ranking customers**. I do not treat the raw score as a perfectly calibrated churn probability.

## 7. Choose a classification threshold

The model can rank customers without a threshold, but a threshold is useful if we want a simple high-risk / low-risk decision.

I choose the threshold using CatBoost's out-of-fold training predictions, and not the test set. F2 is used because it puts more weight on recall than precision.

In [ ]:
threshold_rows = []

for threshold in np.arange(0.01, 0.51, 0.01):
    predictions = (catboost_oof >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_train, predictions, zero_division=0
        ),
        "F2": fbeta_score(
            y_train, predictions, beta=2, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_results.loc[
    threshold_results["F2"].idxmax()
]

selected_threshold = float(best_threshold_row["Threshold"])

print("Selected threshold:", round(selected_threshold, 3))
best_threshold_row.to_frame().T

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    label="Precision"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    label="Recall"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["F2"],
    label="F2"
)

plt.axvline(
    selected_threshold,
    linestyle="--",
    label=f"Selected = {selected_threshold:.2f}"
)

plt.xlabel("Risk-score threshold")
plt.ylabel("Score")
plt.title("Threshold Selection on OOF Predictions")
plt.legend()
plt.show()

## 8. Final test evaluation

Now I fit CatBoost on the full training set and evaluate it once on the held-out test set.

In [ ]:
final_model = CatBoostClassifier(
    **best_catboost_params,
    loss_function="Logloss",
    eval_metric="PRAUC",
    verbose=0,
    random_seed=RANDOM_STATE,
    thread_count=-1
)

final_model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols
)

test_scores = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= selected_threshold).astype(int)

test_metrics = pd.DataFrame({
    "Metric": [
        "ROC AUC",
        "Average Precision",
        "Precision",
        "Recall",
        "F1",
        "F2"
    ],
    "Result": [
        roc_auc_score(y_test, test_scores),
        average_precision_score(y_test, test_scores),
        precision_score(y_test, test_predictions, zero_division=0),
        recall_score(y_test, test_predictions, zero_division=0),
        f1_score(y_test, test_predictions, zero_division=0),
        fbeta_score(y_test, test_predictions, beta=2, zero_division=0)
    ]
})

test_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

PrecisionRecallDisplay.from_predictions(
    y_test,
    test_scores,
    ax=axes[0]
)
axes[0].set_title("Test Precision-Recall Curve")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    ax=axes[1]
)
axes[1].set_title(
    f"Confusion Matrix at Threshold {selected_threshold:.2f}"
)

plt.tight_layout()
plt.show()

## 9. Feature importance

Because the predictors are anonymized, feature importance tells us which variables matter to the model but not what they mean in business terms.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_model.get_feature_importance()
}).sort_values(
    "Importance",
    ascending=False
).head(15)

plt.figure(figsize=(8, 6))

plt.barh(
    feature_importance["Feature"][::-1],
    feature_importance["Importance"][::-1]
)

plt.xlabel("Importance")
plt.title("Top CatBoost Features")
plt.tight_layout()
plt.show()

feature_importance

## 10. Save the model

The final model and a small metadata file are saved for use in later notebooks.

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "catboost_churn_model.cbm"
metadata_path = artifact_dir / "catboost_churn_metadata.json"

final_model.save_model(model_path)

metadata = {
    "selected_threshold": selected_threshold,
    "categorical_features": categorical_cols,
    "feature_names": X_train.columns.tolist(),
    "best_params": best_catboost_params,
    "test_roc_auc": roc_auc_score(y_test, test_scores),
    "test_average_precision": average_precision_score(y_test, test_scores)
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved:", model_path)
print("Saved:", metadata_path)

## Takeaways

- The model is trained only on untreated customers, so the score represents untreated churn risk.
- Logistic regression provides a useful baseline.
- CatBoost is the stronger model for ranking churn risk in this dataset.
- Out-of-fold predictions are used for model comparison and threshold selection.
- The final test set is used only once at the end.
- The CatBoost scores are useful for ranking customers. But they are not perfectly calibrated probabilities.

The next notebook compares this predictive risk ranking with causal treatment-response models.